# Checkpoint + Logvar Comparison: Random Motion Generation

Compare two checkpoints across multiple fixed logvar values for prior-only random motion generation.
Each (checkpoint, logvar) combination is evaluated using the same starting poses.

Uses the **warp** MuJoCo backend with the **no-tail collision** body model.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import time
from pathlib import Path
from typing import Optional, Tuple

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
import orbax.checkpoint as ocp
import pandas as pd
from jax import random
from omegaconf import OmegaConf

from vnl_playground.tasks.rodent import consts as rodent_consts

from scamper.agent.distill.distill_networks import (
    create_abstract_distill_policy,
)
from scamper.agent.mlp_prior.prior_networks import (
    create_abstract_prior_policy,
)
from scamper.agent.observation_utils import flatten_obs_dict
from scamper.scripts.multi_prior_rollout import (
    create_clip_state,
    create_environment,
    create_prior_policy,
    fix_paths,
)

In [2]:
# ============ CHECKPOINT PATHS ============
CHECKPOINT_A_PATH = "/home/mila/a/aidan.sirbu/SCAMPER/scratch/prior_checkpoints/golden_rodent_prior_jax"
CHECKPOINT_A_STEP = 23  # None = latest
CHECKPOINT_A_LABEL = "Prior"
CHECKPOINT_A_TYPE = "prior"  # "prior" or "distill"

CHECKPOINT_B_PATH = "/home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_checkpoints/260327_222349_298131"
CHECKPOINT_B_STEP = None
CHECKPOINT_B_LABEL = "Distill"
CHECKPOINT_B_TYPE = "distill"  # "prior" or "distill"

# ============ ROLLOUT SETTINGS ============
N_STARTING_POSES = 5  # Number of different initial poses (shared across all conditions)
MAX_STEPS = 1000  # Maximum steps per rollout
FIXED_LOGVARS = [-4.0, -3.0, -2.0, -1.0, 0.0]  # List of fixed logvar values to compare
DETERMINISTIC = False  # If True, use mean (no sampling); FIXED_LOGVARS is ignored

# ============ RENDERING ============
M_ROLLOUTS_TO_RENDER = 5  # Number of rollouts to render per condition
RENDER_FPS = 50
CAMERA_NAME = "close_profile-rodent"
RENDER_HEIGHT = 480
RENDER_WIDTH = 640

# ============ RANDOM SEED ============
SEED = 42

## Load Checkpoints

Generic loader that handles both prior (`PriorNetwork`) and distill (`DistillNetwork`) checkpoints.
Both produce policy params in the format `(normalizer_params, {"params": {"encoder": ..., "decoder": ..., "prior": ...}})`.

In [3]:
def load_checkpoint(checkpoint_path: str, step: int | None = None, checkpoint_type: str = "prior"):
    """Load a prior or distill checkpoint.

    Args:
        checkpoint_path: Path to checkpoint directory.
        step: Specific step to load. None = latest.
        checkpoint_type: "prior" or "distill".

    Returns:
        Tuple of (cfg, policy_params, step).
    """
    if checkpoint_type == "prior":
        step_prefix = "PriorNetwork"
    elif checkpoint_type == "distill":
        step_prefix = "DistillNetwork"
    else:
        raise ValueError(f"Unknown checkpoint_type: {checkpoint_type}")

    mgr_options = ocp.CheckpointManagerOptions(create=False, step_prefix=step_prefix)

    with ocp.CheckpointManager(checkpoint_path, options=mgr_options) as ckpt_mgr:
        if step is None:
            step = ckpt_mgr.latest_step()
        print(f"Loading {checkpoint_type} checkpoint from {checkpoint_path} at step {step}")

        cfg = ckpt_mgr.restore(
            step,
            args=ocp.args.Composite(config=ocp.args.JsonRestore()),
        )["config"]
        cfg = OmegaConf.create(cfg)

        # Create abstract policy for restoration (structure depends on checkpoint type)
        if checkpoint_type == "prior":
            prior_hidden_layer_sizes = tuple(
                cfg.network_config.get(
                    "prior_layer_sizes", cfg.network_config.encoder_layer_sizes
                )
            )
            abstract_policy = create_abstract_prior_policy(
                cfg=OmegaConf.to_container(cfg),
                prior_hidden_layer_sizes=prior_hidden_layer_sizes,
            )
        else:  # distill
            obs_sizes = dict(cfg.network_config.obs_sizes)
            abstract_policy = create_abstract_distill_policy(
                obs_sizes=obs_sizes,
                latent_size=int(cfg.network_config.intention_size),
                action_size=int(cfg.network_config.action_size),
                encoder_hidden_layer_sizes=tuple(cfg.network_config.encoder_layer_sizes),
                decoder_hidden_layer_sizes=tuple(cfg.network_config.decoder_layer_sizes),
                prior_hidden_layer_sizes=tuple(cfg.network_config.prior_layer_sizes),
            )

        policy = ckpt_mgr.restore(
            step,
            args=ocp.args.Composite(policy=ocp.args.StandardRestore(abstract_policy)),
        )["policy"]

    cfg = fix_paths(cfg)
    return cfg, policy, step

In [4]:
cfg_a, policy_params_a, step_a = load_checkpoint(CHECKPOINT_A_PATH, CHECKPOINT_A_STEP, CHECKPOINT_A_TYPE)
print(f"  [{CHECKPOINT_A_LABEL}] step={step_a}, action_size={cfg_a.network_config.action_size}, "
      f"intention_size={cfg_a.network_config.intention_size}\n")

cfg_b, policy_params_b, step_b = load_checkpoint(CHECKPOINT_B_PATH, CHECKPOINT_B_STEP, CHECKPOINT_B_TYPE)
print(f"  [{CHECKPOINT_B_LABEL}] step={step_b}, action_size={cfg_b.network_config.action_size}, "
      f"intention_size={cfg_b.network_config.intention_size}")

Loading prior checkpoint from /home/mila/a/aidan.sirbu/SCAMPER/scratch/prior_checkpoints/golden_rodent_prior_jax at step 23
  [Prior] step=23, action_size=38, intention_size=16

Loading distill checkpoint from /home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_checkpoints/260327_222349_298131 at step 49
  [Distill] step=49, action_size=38, intention_size=16


## Create Environment (Warp + No-Tail)

Override the backend to warp and use the no-tail collision XML, then create the environment.

In [5]:
# Use checkpoint A's config as base, override backend and XML
env_cfg = OmegaConf.create(OmegaConf.to_container(cfg_a, resolve=True))
OmegaConf.set_struct(env_cfg.env_config, False)
OmegaConf.update(env_cfg.env_config, "mujoco_impl", "warp")
OmegaConf.update(
    env_cfg.env_config,
    "walker_xml_path",
    str(rodent_consts.RODENT_NO_TAIL_COLLISION_XML),
)
OmegaConf.set_struct(env_cfg.env_config, True)

print(f"Backend: {env_cfg.env_config.mujoco_impl}")
print(f"Walker XML: {env_cfg.env_config.walker_xml_path}")

env = create_environment(env_cfg)

# Access reference clips for sampling starting poses
unwrapped_env = env
while hasattr(unwrapped_env, "_env"):
    unwrapped_env = unwrapped_env._env

reference_clips = unwrapped_env.reference_clips
num_clips = reference_clips.qpos.shape[0]
clip_length = reference_clips.qpos.shape[1]

print(f"Reference clips: {num_clips} clips, {clip_length} frames each")
print(f"Action size: {env.action_size}")

Backend: warp
Walker XML: /home/mila/a/aidan.sirbu/vnl-playground/vnl_playground/tasks/rodent/xmls/rodent_no_tail_collisions.xml


KeyboardInterrupt: 

## Create Policy Functions

Build a prior-only policy for each (checkpoint, logvar) combination.

In [ ]:
def get_network_params(cfg):
    """Extract network architecture params from a checkpoint config."""
    nc = cfg.network_config
    action_size = nc.action_size
    intention_size = nc.intention_size
    decoder_layers = tuple(nc.decoder_layer_sizes)
    prior_layers = tuple(nc.get("prior_layer_sizes", nc.encoder_layer_sizes))

    if "obs_sizes" in nc:
        proprio_obs_size = nc.obs_sizes["proprioception"]
    elif "proprioceptive_obs_size" in nc:
        proprio_obs_size = nc.proprioceptive_obs_size
    else:
        proprio_obs_size = nc.observation_size - nc.get("reference_obs_size", 0)

    return action_size, intention_size, decoder_layers, prior_layers, proprio_obs_size


def make_policy(cfg, policy_params, fixed_logvar, deterministic):
    """Create a prior-only policy function from a checkpoint and logvar setting."""
    action_size, intention_size, decoder_layers, prior_layers, proprio_obs_size = (
        get_network_params(cfg)
    )
    return create_prior_policy(
        policy_params=policy_params,
        action_size=action_size,
        proprioceptive_obs_size=proprio_obs_size,
        intention_latent_size=intention_size,
        decoder_hidden_layer_sizes=decoder_layers,
        prior_hidden_layer_sizes=prior_layers,
        fixed_logvar=fixed_logvar if not deterministic else None,
        deterministic=deterministic,
    )


# Build (checkpoint_label, logvar) -> policy_fn mapping
checkpoints = {
    CHECKPOINT_A_LABEL: (cfg_a, policy_params_a),
    CHECKPOINT_B_LABEL: (cfg_b, policy_params_b),
}

if DETERMINISTIC:
    logvar_labels = ["deterministic"]
    logvar_values = [None]
else:
    logvar_labels = [f"logvar={lv}" for lv in FIXED_LOGVARS]
    logvar_values = FIXED_LOGVARS

# policy_fns[(ckpt_label, logvar_label)] = policy_fn
policy_fns = {}
for ckpt_label, (ckpt_cfg, ckpt_params) in checkpoints.items():
    net_params = get_network_params(ckpt_cfg)
    print(f"[{ckpt_label}] action={net_params[0]}, intention={net_params[1]}, "
          f"proprio={net_params[4]}, decoder={net_params[2]}, prior={net_params[3]}")
    for lv, lv_label in zip(logvar_values, logvar_labels):
        policy_fns[(ckpt_label, lv_label)] = make_policy(ckpt_cfg, ckpt_params, lv, DETERMINISTIC)

mode_str = "deterministic" if DETERMINISTIC else f"stochastic, logvars={FIXED_LOGVARS}"
print(f"\nSampling mode: {mode_str}")
print(f"Total conditions: {len(policy_fns)} ({len(checkpoints)} checkpoints x {len(logvar_labels)} logvar values)")

[Prior] action=38, intention=16, proprio=277, decoder=(1024, 1024, 512), prior=(1536, 1024, 512)
[Distill] action=38, intention=16, proprio=277, decoder=(1024, 1024, 512), prior=(1536, 1024, 512)

Sampling mode: stochastic, logvars=[-4.0, -3.0, -2.0, -1.0, 0.0]
Total conditions: 10 (2 checkpoints x 5 logvar values)


## Sample Starting Poses and Run Rollouts

Create all starting states once, then run each rollout for every (checkpoint, logvar) combination.
The same starting poses and per-pose RNG keys are used across all conditions.

In [ ]:
# Sample n random starting poses from the reference clips
rng_np = np.random.default_rng(SEED)
clip_indices = rng_np.integers(0, num_clips, size=N_STARTING_POSES)
start_frames = rng_np.integers(0, clip_length, size=N_STARTING_POSES)
pose_configs = list(zip(clip_indices, start_frames))

print(f"Sampled {N_STARTING_POSES} starting poses:")
for i, (ci, sf) in enumerate(pose_configs[:5]):
    print(f"  Pose {i}: clip={ci}, frame={sf}")
if N_STARTING_POSES > 5:
    print(f"  ... ({N_STARTING_POSES - 5} more)")

# Create all starting states upfront (shared across all conditions)
jax_rng = random.PRNGKey(SEED)
starting_states_list = []
pose_rng_keys = []

for clip_idx, start_frame in pose_configs:
    jax_rng, rng_init, rng_rollout = random.split(jax_rng, 3)
    pose_rng_keys.append(rng_rollout)
    starting_states_list.append(create_clip_state(env, int(clip_idx), int(start_frame), rng_init))

print(f"Created {N_STARTING_POSES} starting states")


def create_rollout_fn(env, policy_fn, max_steps):
    """Create a JIT-compiled rollout fn that takes (starting_state, rng_key) as args."""
    jit_step = jax.jit(env.step)

    @jax.jit
    def rollout_fn(starting_state, rng_key):
        state = starting_state

        def step_fn(carry, _):
            state, key = carry
            key, key_action = random.split(key)

            if hasattr(state.obs, "get") or isinstance(state.obs, dict):
                flat_obs = flatten_obs_dict(state.obs)
                proprio = flat_obs["proprioception"]
            else:
                proprio = state.obs

            action, _ = policy_fn(proprio, key_action)
            next_state = jit_step(state, action)

            return (next_state, key), next_state

        initial_carry = (state, rng_key)
        _, all_states = jax.lax.scan(
            step_fn, initial_carry, None, length=max_steps
        )

        return all_states

    return rollout_fn


# Run all rollouts: 1 compilation per (checkpoint, logvar), then n fast executions
# results[(ckpt_label, logvar_label)] = {"qpos": ...}
results = {}

total_start = time.time()

for (ckpt_label, lv_label), policy_fn in policy_fns.items():
    condition = f"{ckpt_label} / {lv_label}"
    print(f"\nRunning {N_STARTING_POSES} rollouts for [{condition}]...")
    t0 = time.time()

    rollout_fn = create_rollout_fn(env, policy_fn, MAX_STEPS)

    all_qpos = []

    for pose_idx in range(N_STARTING_POSES):
        rng_key = pose_rng_keys[pose_idx]
        states = rollout_fn(starting_states_list[pose_idx], rng_key)
        states.data.qpos.block_until_ready()

        all_qpos.append(np.array(states.data.qpos))
        del states

        if pose_idx == 0:
            print(f"  Compiled + ran first rollout in {time.time() - t0:.1f}s")

    results[(ckpt_label, lv_label)] = {
        "qpos": np.stack(all_qpos),  # (n, max_steps, qpos_dim)
    }
    elapsed_label = time.time() - t0
    print(f"  Done in {elapsed_label:.1f}s")

print(f"\nTotal: {time.time() - total_start:.1f}s")

## Statistics

Compute integrated distance travelled and other metrics across rollouts.

In [ ]:
def calculate_distance_travelled(qpos: np.ndarray) -> np.ndarray:
    """Calculate total XY distance travelled for each rollout.

    Args:
        qpos: Shape (num_rollouts, max_steps, qpos_dim).

    Returns:
        Total distance per rollout, shape (num_rollouts,).
    """
    xy = qpos[:, :, :2]
    diffs = np.diff(xy, axis=1)
    step_distances = np.linalg.norm(diffs, axis=2)
    return np.sum(step_distances, axis=1)


# Build comparison DataFrame
rows = []
for (ckpt_label, lv_label), res in results.items():
    full_dist = calculate_distance_travelled(res["qpos"])
    for i in range(N_STARTING_POSES):
        rows.append({
            "checkpoint": ckpt_label,
            "logvar": lv_label,
            "condition": f"{ckpt_label} / {lv_label}",
            "pose_idx": i,
            "clip_idx": int(pose_configs[i][0]),
            "start_frame": int(pose_configs[i][1]),
            "distance": full_dist[i],
        })

df = pd.DataFrame(rows)

# Summary statistics
summary = df.groupby(["checkpoint", "logvar"]).agg(
    distance_mean=("distance", "mean"),
    distance_std=("distance", "std"),
).round(2)

print("Summary Statistics")
print("=" * 80)
print(summary.to_string())

In [ ]:
# Histograms: one row per logvar, columns = distance histogram / paired scatter
n_logvars = len(logvar_labels)
fig, axes = plt.subplots(n_logvars, 2, figsize=(12, 5 * n_logvars), squeeze=False)

ckpt_labels = [CHECKPOINT_A_LABEL, CHECKPOINT_B_LABEL]

for row, lv_label in enumerate(logvar_labels):
    # 1. Distance travelled
    ax = axes[row, 0]
    for ckpt_label in ckpt_labels:
        subset = df[(df["checkpoint"] == ckpt_label) & (df["logvar"] == lv_label)]
        ax.hist(subset["distance"], alpha=0.6, bins=20, label=ckpt_label, edgecolor="black")
    ax.set_xlabel("Distance (m)")
    ax.set_ylabel("Count")
    ax.set_title(f"Integrated Distance ({lv_label})")
    ax.legend()

    # 2. Paired scatter (checkpoint A vs B)
    ax = axes[row, 1]
    dist_a = df[(df["checkpoint"] == CHECKPOINT_A_LABEL) & (df["logvar"] == lv_label)]["distance"].values
    dist_b = df[(df["checkpoint"] == CHECKPOINT_B_LABEL) & (df["logvar"] == lv_label)]["distance"].values
    ax.scatter(dist_a, dist_b, alpha=0.7, s=40, edgecolors="black", linewidth=0.5)
    max_val = max(dist_a.max(), dist_b.max()) * 1.1
    ax.plot([0, max_val], [0, max_val], "k--", alpha=0.4, label="y=x")
    ax.set_xlabel(f"{CHECKPOINT_A_LABEL} Distance (m)")
    ax.set_ylabel(f"{CHECKPOINT_B_LABEL} Distance (m)")
    ax.set_title(f"Paired Distance ({lv_label})")
    ax.set_aspect("equal")
    ax.legend()

plt.tight_layout()
plt.show()

## XY Trajectory Plots

Origin-centered XY trajectories for all rollouts, side by side.

In [ ]:
# XY trajectory plots: grid of (n_logvars rows) x (2 checkpoint columns)
n_logvars = len(logvar_labels)
ckpt_labels = [CHECKPOINT_A_LABEL, CHECKPOINT_B_LABEL]
fig, axes = plt.subplots(n_logvars, 2, figsize=(12, 6 * n_logvars), squeeze=False)

for row, lv_label in enumerate(logvar_labels):
    for col, ckpt_label in enumerate(ckpt_labels):
        ax = axes[row, col]
        key = (ckpt_label, lv_label)
        qpos = results[key]["qpos"]
        for i in range(N_STARTING_POSES):
            xy = qpos[i, :, :2]
            xy = xy - xy[0:1, :]  # origin-center
            ax.plot(xy[:, 0], xy[:, 1], alpha=0.5, linewidth=0.8)
        ax.plot(0, 0, "ko", markersize=4, zorder=5)
        ax.set_aspect("equal")
        ax.set_title(f"{ckpt_label} / {lv_label}")
        ax.set_xlabel("X (m)")
        ax.set_ylabel("Y (m)")
        ax.grid(True, alpha=0.2)

# Shared axis limits
all_xy = np.concatenate([
    res["qpos"][:, :, :2].reshape(-1, 2) for res in results.values()
])
lim = np.abs(all_xy - all_xy.mean(axis=0)).max() * 1.1
for ax in axes.flat:
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

plt.tight_layout()
plt.show()

## Side-by-Side Video Rendering

For each starting pose, render all (checkpoint, logvar) conditions side by side.
Each row in the video grid is a logvar value; each column is a checkpoint.

In [ ]:
def render_qpos_trajectory(qpos_trajectory, mj_model, camera_name, height=480, width=640):
    """Render frames directly from a qpos trajectory using MuJoCo's C renderer."""
    mj_data = mujoco.MjData(mj_model)
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    frames = []
    for qpos in qpos_trajectory:
        mj_data.qpos[:] = qpos
        mujoco.mj_forward(mj_model, mj_data)
        renderer.update_scene(mj_data, camera=camera_name)
        frames.append(renderer.render().copy())

    renderer.close()
    return frames


# Render first M_ROLLOUTS_TO_RENDER poses
render_indices = list(range(min(M_ROLLOUTS_TO_RENDER, N_STARTING_POSES)))

ckpt_labels = [CHECKPOINT_A_LABEL, CHECKPOINT_B_LABEL]
n_logvars = len(logvar_labels)
n_ckpts = len(ckpt_labels)

print(f"Rendering {len(render_indices)} rollouts as grids "
      f"({n_logvars} logvar rows x {n_ckpts} checkpoint columns)")
print(f"Pose indices: {render_indices}")

mj_model = env.mj_model

for render_idx, pose_idx in enumerate(render_indices):
    clip_idx, start_frame = pose_configs[pose_idx]
    print(f"\nRollout {render_idx + 1}/{len(render_indices)}: "
          f"pose {pose_idx} (clip={clip_idx}, frame={start_frame})")

    # Render all conditions for this pose
    all_frames = {}  # (ckpt_label, lv_label) -> list of frames
    for lv_label in logvar_labels:
        for ckpt_label in ckpt_labels:
            key = (ckpt_label, lv_label)
            qpos = results[key]["qpos"][pose_idx]

            t0 = time.time()
            frames = render_qpos_trajectory(
                qpos, mj_model, CAMERA_NAME, RENDER_HEIGHT, RENDER_WIDTH,
            )
            all_frames[key] = frames
            print(f"  [{ckpt_label} / {lv_label}] {len(frames)} frames in "
                  f"{time.time() - t0:.1f}s")

    # Assemble grid: rows=logvars, cols=checkpoints
    n_frames = len(next(iter(all_frames.values())))
    combined = []
    for frame_idx in range(n_frames):
        rows = []
        for lv_label in logvar_labels:
            row = np.concatenate(
                [all_frames[(ckpt_label, lv_label)][frame_idx] for ckpt_label in ckpt_labels],
                axis=1,
            )
            rows.append(row)
        combined.append(np.concatenate(rows, axis=0))

    media.show_video(combined, fps=RENDER_FPS)